# SCD2 Real-Time Data Processing

**Parameter:** Use the `catalog` parameter widget above to specify the target catalog for the SCD2 table.

**Default:** `scdtypes`

**Target Table:** `{catalog}.scd_realtime.scd2_realtime_data`

## Step 1: Create Target Dataset (1 Million Records)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

target = (
    spark.range(1,1000001)
    .withColumnRenamed("id","customer_id")

    .withColumn("first_name",concat(lit("First_"),col("customer_id")))
    .withColumn("last_name",concat(lit("Last_"),col("customer_id")))

    .withColumn("email",
                concat(col("customer_id"),lit("@gmail.com")))

    .withColumn("phone",
                concat(lit("98"),lpad(col("customer_id"),8,"0")))

    .withColumn("gender",
                when(col("customer_id")%2==0,"Male")
                .otherwise("Female"))

    .withColumn("city",
                when(col("customer_id")%5==0,"Hyderabad")
                .when(col("customer_id")%5==1,"Bangalore")
                .when(col("customer_id")%5==2,"Chennai")
                .when(col("customer_id")%5==3,"Mumbai")
                .otherwise("Delhi"))

    .withColumn("state",
                when(col("customer_id")%5==0,"TS")
                .when(col("customer_id")%5==1,"KA")
                .when(col("customer_id")%5==2,"TN")
                .when(col("customer_id")%5==3,"MH")
                .otherwise("DL"))

    .withColumn("country",lit("India"))

    .withColumn("salary",(col("customer_id")*10)+50000)

    .withColumn("department",
                when(col("customer_id")%3==0,"IT")
                .when(col("customer_id")%3==1,"HR")
                .otherwise("Finance"))

    .withColumn(
        "effective_date",
        expr("timestamp('2025-01-01 09:00:00')") + (col("customer_id") % 365).cast("interval day")
    )
)

In [0]:
# display(target)

## Step 2: Add SCD Columns

In [0]:
target = target \
.withColumn("start_date",col("effective_date")) \
.withColumn("end_date",lit("9999-12-31").cast("timestamp")) \
.withColumn("is_active",lit("Y"))

In [0]:
# display(target)

## Step 3: Create Source Dataset

In [0]:
source = target.drop(
    "start_date",
    "end_date",
    "is_active"
)

In [0]:
# display(source)

## Step 4: Simulate 20,000 Updated Records

Update the city and effective date.

In [0]:
source = source \
.withColumn(
    "city",
    when(
        (col("customer_id")>=400001) &
        (col("customer_id")<=420000),
        "Pune"
    ).otherwise(col("city"))
) \
.withColumn(
    "effective_date",
    when(
        (col("customer_id")>=400001) &
        (col("customer_id")<=420000),
        lit("2026-08-10 11:30:00").cast("timestamp")
    ).otherwise(col("effective_date"))
)

In [0]:
# display(source)

## Step 5: Insert 10,000 New Customers

In [0]:
new_customers = (
spark.range(1000001,1010001)

.withColumnRenamed("id","customer_id")

.withColumn("first_name",concat(lit("First_"),col("customer_id")))
.withColumn("last_name",concat(lit("Last_"),col("customer_id")))

.withColumn("email",
concat(col("customer_id"),lit("@gmail.com")))

.withColumn("phone",
concat(lit("98"),lpad(col("customer_id"),8,"0")))

.withColumn("gender",lit("Male"))

.withColumn("city",lit("Hyderabad"))

.withColumn("state",lit("TS"))

.withColumn("country",lit("India"))

.withColumn("salary",lit(70000))

.withColumn("department",lit("IT"))

.withColumn(
"effective_date",
lit("2026-08-10 10:00:00").cast("timestamp")
)
)

In [0]:
# display(new_customers)

## Append them:

In [0]:
source = source.unionByName(new_customers)
# display(source.count())
# display(source)

## Step 6: Generate Hash

In [0]:
source = source.withColumn(
    "hash_key",
    xxhash64(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "gender",
        "city",
        "state",
        "country",
        "salary",
        "department"
    )
)

target = target.withColumn(
    "hash_key",
    xxhash64(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "gender",
        "city",
        "state",
        "country",
        "salary",
        "department"
    )
)
# display(source)
# display(target)

## Step 7 — Find New + Changed Records Using LEFT ANTI

In [0]:
from pyspark.sql.functions import col

target_active = target.filter(
    col("is_active") == "Y"
)

changed_new = (
    source.alias("s")
    .join(
        target_active.alias("t"),
        (col("s.customer_id") == col("t.customer_id")) &
        (col("s.hash_key") == col("t.hash_key")),
        "leftanti"
    )
)
# display(changed_new)

## Step 8 — Prepare SCD Columns for New/Changed Records

In [0]:
changed_new = (
    changed_new
    .withColumn("start_date", col("effective_date"))
    .withColumn(
        "end_date",
        lit("9999-12-31").cast("timestamp")
    )
    .withColumn("is_active", lit("Y"))
)
# display(changed_new)

## Step 9 — Combine Existing Target History + New Versions

In [0]:
history_df = target.unionByName(changed_new)
# display(history_df)

## Step 10 — Use ROW_NUMBER()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("start_date").desc())
)

history_df = history_df.withColumn(
    "rn",
    row_number().over(window_spec)
)
# display(history_df)

## Step 11 — Use LAG()

In [0]:
from pyspark.sql.functions import lag

window_lag = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("rn"))
)

history_df = history_df.withColumn(
    "next_start_date",
    lag("start_date").over(window_lag)
)
# display(history_df)

## Step 12 — Update END_DATE

In [0]:
from pyspark.sql.functions import when

history_df = history_df.withColumn(
    "end_date",
    when(
        col("rn") > 1,
        col("next_start_date")
    ).otherwise(
        col("end_date")
    )
)
# display(history_df)

## Step 13 — Update IS_ACTIVE

In [0]:
history_df = history_df.withColumn(
    "is_active",
    when(col("rn") == 1, "Y")
    .otherwise("N")
)
# display(history_df)

## Step 14 — Remove Temporary Columns

In [0]:
final_scd2 = history_df.drop(
    "rn",
    "next_start_date",
    "hash_key"
)
# display(final_scd2)

In [0]:
# # Get catalog parameter
# catalog = dbutils.widgets.get("catalog")

# # Create schema if it doesn't exist
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.scd_realtime")

In [0]:
# Get catalog parameter
catalog = dbutils.widgets.get("catalog")

# Write to parameterized table
final_scd2.write.mode("overwrite").format("delta").option("mergeSchema","true").saveAsTable(f"{catalog}.scd_realtime.scdtype2_realtime_data")

In [0]:
# display(final_scd2.filter(col("email")=='400013@gmail.com'))

In [0]:
# %sql
# select * from ${catalog}.scd_realtime.scd2_realtime_data;

In [0]:
# %sql
# select * from ${catalog}.scd_realtime.scd2_realtime_data where email="400013@gmail.com";